In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

In [8]:
df = pd.read_csv('data/COVID_US_cases.csv', parse_dates=['date'])
df = df[df['new_confirmed'] >= 0].copy()
df = df[(df['date'] >= '2020-01-01') & (df['date'] <= '2022-09-15')].copy()
df['year'] = df['date'].dt.year
df['week'] = df['date'].dt.isocalendar().week.astype(int)

weekly = (
    df.groupby(['year', 'week'])
    .agg(cases=('new_confirmed', 'sum'), date=('date', 'min'))
    .reset_index()
)

pivot = weekly.pivot(index='year', columns='week', values='cases')
pivot = pivot.reindex(columns=range(1, 54), fill_value=np.nan)
last_2022_week = weekly[weekly['year'] == 2022]['week'].max()
pivot.loc[2022, last_2022_week + 1:] = np.nan

pivot

week,1,2,3,4,5,6,7,8,9,10,...,44,45,46,47,48,49,50,51,52,53
year,,,,,,,,,,,,,,,,,,,,,
2020,NaN,NaN,1.0,9.0,5.0,7.0,6.0,16.0,74.0,716.0,...,636056.0,803264.0,1050512.0,1203780.0,1127257.0,1383427.0,1488601.0,1507244.0,1274666.0,820170.0
2021,1714127.0,1515267.0,1185176.0,1026227.0,820731.0,632066.0,450854.0,470888.0,405346.0,422015.0,...,515022.0,587572.0,655522.0,502616.0,849875.0,837105.0,932649.0,1439094.0,2438729.0,671431.0
2022,5036233.0,5648074.0,4849622.0,3656770.0,2089957.0,1209394.0,748061.0,456386.0,333722.0,244471.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,448828.0,NaN


In [9]:
# Layout: each year = 4-col x 13-row block of square cells (4 weeks/row ≈ monthly)
COLS_PER_YEAR = 4
ROWS_PER_YEAR = 13
GAP = 1.2

years = sorted(pivot.index.tolist())
n_years = len(years)

cmap = mcolors.LinearSegmentedColormap.from_list(
    'white_red', ['#ffffff', '#fee0d2', '#fc9272', '#de2d26', '#67000d']
)
max_cases = weekly['cases'].max()

# Quarter labels only: each quarter ≈ 13 weeks = 3.25 rows.
# Place at the first row of each quarter (rows 0, 3, 6, 9).
quarter_row_labels = {0: 'Q1 (Jan)', 3: 'Q2 (Apr)', 6: 'Q3 (Jul)', 9: 'Q4 (Oct)'}

total_x = n_years * COLS_PER_YEAR + (n_years - 1) * GAP
total_y = ROWS_PER_YEAR

cell_size    = 0.55
left_margin  = 1.0
right_margin = 0.5
top_margin   = 1.0
bottom_margin = 1.1

fig_w = left_margin + total_x * cell_size + right_margin
fig_h = top_margin  + total_y * cell_size + bottom_margin

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_aspect('equal')
ax.set_xlim(0, total_x)
ax.set_ylim(0, total_y)
ax.axis('off')

def year_x_offset(year_idx):
    return year_idx * (COLS_PER_YEAR + GAP)

# Draw cells
for yi, year in enumerate(years):
    x0 = year_x_offset(yi)
    for week in range(1, 54):
        row = (week - 1) // COLS_PER_YEAR
        col = (week - 1) % COLS_PER_YEAR
        if row >= ROWS_PER_YEAR:
            continue
        cases = pivot.loc[year, week] if week in pivot.columns else np.nan
        color = '#eeeeee' if pd.isna(cases) else cmap(cases / max_cases)
        rect = Rectangle(
            (x0 + col, total_y - 1 - row), 1, 1,
            facecolor=color, edgecolor='white', linewidth=0.8
        )
        ax.add_patch(rect)

# Year labels above each block
for yi, year in enumerate(years):
    x0 = year_x_offset(yi)
    cx = x0 + COLS_PER_YEAR / 2
    ax.text(cx, total_y + 0.25, str(year),
            ha='center', va='bottom', fontsize=12,
            fontweight='bold', color='#222222')

# W1–W4 column headers
for col in range(COLS_PER_YEAR):
    ax.text(year_x_offset(0) + col + 0.5, total_y + 0.05,
            f'W{col+1}', ha='center', va='bottom',
            fontsize=7, color='#888888')

# Quarter row labels (only 4 labels, no drift problem)
for row_i, label in quarter_row_labels.items():
    y = total_y - row_i - 0.5
    ax.text(-0.2, y, label, ha='right', va='center',
            fontsize=8.5, color='#444444')
    # Small tick line to anchor the label
    ax.plot([0, -0.1], [y, y], color='#cccccc', linewidth=0.7,
            transform=ax.transData, clip_on=False)

# Colorbar (cases)
sm = plt.cm.ScalarMappable(cmap=cmap,
                            norm=mcolors.Normalize(vmin=0, vmax=max_cases))
sm.set_array([])
cbar_ax = fig.add_axes([0.12, 0.04, 0.50, 0.025])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label('New confirmed cases per week (US)', fontsize=9, color='#333333')
cbar.ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{int(x/1_000_000)}M' if x >= 1_000_000
                      else (f'{int(x/1000)}k' if x >= 1000 else str(int(x))))
)
cbar.ax.tick_params(labelsize=8, colors='#444444')
cbar.outline.set_edgecolor('#cccccc')

# No-data swatch to the right of the colorbar
nd_ax = fig.add_axes([0.64, 0.04, 0.025, 0.025])
nd_ax.set_xlim(0, 1)
nd_ax.set_ylim(0, 1)
nd_ax.add_patch(Rectangle((0, 0), 1, 1, facecolor='#eeeeee',
                           edgecolor='#cccccc', linewidth=0.8))
nd_ax.axis('off')
fig.text(0.676, 0.052, 'No data', va='center', fontsize=8, color='#555555')

# Title
fig.text(0.5, 0.985, 'COVID-19 Cases in the United States, 2020\u20132022',
         ha='center', va='top', fontsize=13, fontweight='bold', color='#111111')
fig.text(0.5, 0.955, 'Each cell = one week. Rows \u2248 months (4 weeks each). Color intensity shows new confirmed cases.',
         ha='center', va='top', fontsize=8.5, color='#555555')

plt.savefig('assets/final.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved to assets/final.png')

Saved to assets/final.png
